# trustcv v1.1.0 — Making Trust Claims Correct

This notebook walks through what changed in trustcv v1.1.0 and why it matters.

trustcv's job is to tell you whether a cross-validated performance estimate can be trusted.
Before v1.1.0, several of its checks could say **"PASSED"** in situations where nothing had
actually been verified, or where the estimate was badly wrong. v1.1.0 fixes this by making
every check report one of three honest states — `PASSED`, `FAILED`, or `NOT_CHECKED` — and by
introducing `recommend_cv`, a small advisor that suggests a splitting strategy for your data.

We'll cover three things:

1. The classic leakage mistake (feature selection before cross-validation) — and how v1.1.0
   reports it honestly instead of hiding it.
2. The correct workflow (an sklearn `Pipeline`) — showing that trustcv still gives a clean
   "PASSED" when nothing is wrong.
3. `recommend_cv` — asking trustcv which cross-validation method fits your data.

If you're upgrading from v1.0.7, see `CHANGELOG.md` for the full list of behaviour changes.

In [1]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")  # sklearn convergence warnings, not relevant here

from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from trustcv import TrustCV, recommend_cv

print("trustcv is ready.")

trustcv is ready.


## 1. The classic mistake: feature selection before cross-validation

We build a dataset of **pure noise** — 100 samples, 5,000 random features, and a random
binary label. There is, by construction, no real signal at all: the true accuracy of any
model here is 50%.

Then we make the classic mistake: we run `SelectKBest` on the **entire** dataset before
handing it to cross-validation. Because the selection step has already seen every label,
including the ones that end up in each test fold, the reported score becomes badly inflated.

In [2]:
rng = np.random.default_rng(0)
X = rng.normal(size=(100, 5000))
y = rng.integers(0, 2, 100)

# The leak: feature selection sees the full dataset, test folds included.
X_selected = SelectKBest(k=20).fit_transform(X, y)

result = TrustCV(method="stratified_kfold", n_splits=5, random_state=0).validate(
    model=LogisticRegression(max_iter=2000), X=X_selected, y=y
)

print(f"Reported ROC-AUC: {result.mean_scores['roc_auc']:.2f}")
print(f"Overall status:   {result.overall_status}")
print()
print(result.summary())

Reported ROC-AUC: 0.96
Overall status:   NOT_FULLY_VERIFIED

=== Trustworthy Cross-Validation Results ===

Performance Metrics (mean +/- std) (method: corrected_t):
  accuracy: 0.870 +/- 0.057 [95% CI (corrected_t): 0.764-0.976]
  roc_auc: 0.956 +/- 0.037 [95% CI (corrected_t): 0.887-1.024]
  sensitivity: 0.927 +/- 0.076 [95% CI (corrected_t): 0.786-1.069]
  specificity: 0.800 +/- 0.122 [95% CI (corrected_t): 0.573-1.027]
  precision: 0.856 +/- 0.072 [95% CI (corrected_t): 0.721-0.991]
  recall: 0.927 +/- 0.076 [95% CI (corrected_t): 0.786-1.069]
  f1: 0.887 +/- 0.048 [95% CI (corrected_t): 0.798-0.976]

Data Integrity Checks:
  Duplicate Samples: PASSED — No exact duplicate feature rows were found.
  Group Leakage: NOT_CHECKED — No groups or patient_ids were supplied, so group leakage cannot be checked.
  Preprocessing Leakage: NOT_CHECKED — Cannot verify that preprocessing (scaling, imputation, feature selection) was fit only on training folds. Wrap preprocessing in an sklearn Pipeli

**What to notice:**

- The reported AUC is close to 0.95, even though the true AUC is 0.50. The inflation comes
  entirely from the feature selection step, not from anything trustcv's splitter did wrong.
- `Preprocessing Leakage` reads **`NOT_CHECKED`**, not `PASSED`. trustcv cannot see what
  happened to the data before `validate()` was called, so it refuses to claim otherwise.
- `overall_status` is **not** `PASSED`. In v1.0.7, this same code would have printed
  `Leakage Check: PASSED`, which was actively misleading.

The fix is not a trustcv setting — it's a change to how the model is built. Preprocessing
that touches the label (like `SelectKBest`) has to be refit separately inside every fold.

## 2. The correct workflow: preprocessing inside a Pipeline

Now we do it properly. We wrap the feature selection and the classifier together in a single
`sklearn.pipeline.Pipeline`. trustcv refits the whole pipeline — selection included — on each
training fold only, so no information from the test fold leaks into the selection step.

In [3]:
correct_model = make_pipeline(SelectKBest(k=20), LogisticRegression(max_iter=2000))

result_correct = TrustCV(method="stratified_kfold", n_splits=5, random_state=0).validate(
    model=correct_model, X=X, y=y  # note: raw X, not the pre-selected X_selected
)

print(f"Reported ROC-AUC: {result_correct.mean_scores['roc_auc']:.2f}  (truth: 0.50)")
print(f"Overall status:   {result_correct.overall_status}")

Reported ROC-AUC: 0.44  (truth: 0.50)
Overall status:   NOT_FULLY_VERIFIED


The AUC now comes out close to 0.50, which matches the truth: there is no real signal in
this data. This is the honest result the earlier, leaky version was hiding.

Now let's check that trustcv also gives a clean bill of health on real data, when the
workflow is correct and there's nothing to flag. We'll use the Breast Cancer Wisconsin
dataset with a `StandardScaler` + `LogisticRegression` pipeline, and tell trustcv that our
samples are independent (there are no repeated patients or groups in this dataset).

In [4]:
X_bc, y_bc = load_breast_cancer(return_X_y=True)
clean_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

result_clean = TrustCV(
    method="stratified_kfold", n_splits=5, random_state=0,
    declare_independent_samples=True,
).validate(model=clean_model, X=X_bc, y=y_bc)

print(f"ROC-AUC: {result_clean.mean_scores['roc_auc']:.3f}  "
      f"95% CI: {result_clean.confidence_intervals['roc_auc']}")
print(f"Overall status: {result_clean.overall_status}")

ROC-AUC: 0.995  95% CI: (0.9837629622781344, 1.0071486573101964)
Overall status: PASSED


`overall_status` is now `PASSED` — every applicable check ran and came back clean. This is
what a genuinely trustworthy result looks like: high performance **and** every check that
could be run for it.

**A note on the confidence interval.** By default, trustcv now uses the Nadeau–Bengio
*corrected* t-interval rather than a plain bootstrap over fold scores. The old default
under-covered badly: on the pure-noise example above, its 95% interval contained the true
value (0.50) only about 70% of the time, instead of 95%. The new default fixes that. You can
still request the legacy method with `ci_method="bootstrap"` if you need it, but trustcv
will warn you that it tends to be overconfident.

## 3. Asking trustcv what to do: `recommend_cv`

Choosing the right cross-validation strategy is often the hardest part — should you group by
patient? Split by time? By location? `recommend_cv` looks at what you pass it and suggests a
method, along with a runnable splitter and an explanation.

Here we simulate a dataset with **repeated groups**: 60 "patients", each contributing 4 rows,
with a fairly rare positive class (about 20% of patients).

In [5]:
rng = np.random.default_rng(7)
groups = np.repeat(np.arange(60), 4)
y_grouped = np.repeat((rng.random(60) < 0.2).astype(int), 4)
X_grouped = rng.normal(size=(240, 5))

recommendation = recommend_cv(X_grouped, y_grouped, groups=groups)

print("Category: ", recommendation.category)
print("Splitter: ", type(recommendation.splitter).__name__)
print()
print("Rationale:")
print(" ", recommendation.rationale)
if recommendation.warnings:
    print()
    print("Warnings:")
    for w in recommendation.warnings:
        print(" -", w)
print()
print("Suggested code:")
print(recommendation.code)

Category:  grouped
Splitter:  StratifiedGroupKFold

Rationale:
  Repeated group IDs show that rows are not independent; stratified group folds keep each subject together while preserving class prevalence as far as possible.

Suggested code:
from trustcv import StratifiedGroupKFold
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)



`recommend_cv` correctly identifies that rows are grouped (repeated patient IDs) and
recommends a stratified group splitter, so that no patient's rows end up split across both
the training and test folds. The returned `splitter` is ready to use directly:

In [6]:
result_grouped = TrustCV(method="kfold").validate(
    model=LogisticRegression(max_iter=2000),
    X=X_grouped, y=y_grouped, groups=groups,
    cv=recommendation.splitter,
)

print(f"Group leakage check: {result_grouped.checks['group_leakage'].status}")
print(f"Overall status:      {result_grouped.overall_status}")

Group leakage check: PASSED
Overall status:      NOT_FULLY_VERIFIED


Notice that `overall_status` here is `NOT_FULLY_VERIFIED`, not `PASSED` — this toy example
uses a raw estimator with no preprocessing `Pipeline` and no permutation check enabled, so
those two checks stay `NOT_CHECKED`. The point of this example is narrower: it shows that
`group_leakage` specifically reports `PASSED` once the recommended group-aware splitter is
used, instead of silently ignoring the groups.

## Summary

| Situation | v1.0.7 | v1.1.0 |
|---|---|---|
| Preprocessing leak before `validate()` | Could report `PASSED` | Reports `NOT_CHECKED`; overall status can't be `PASSED` |
| No groups supplied | Reported `PASSED` | Reports `NOT_CHECKED` (or `NOT_APPLICABLE` if you declare independence) |
| Leakage detector crashes | Silently treated as passed | Reported as `ERROR` |
| 95% CI on noise data | ~70% coverage | ~95% coverage (corrected t / OOF bootstrap) |
| Two unrelated random samples | Could be flagged "near-duplicate" | Not flagged |
| Choosing a CV method | Manual, or `suggest_best_method` (kept, but limited) | `recommend_cv` — full rationale, warnings, runnable code |

See `CHANGELOG.md` for the complete list of changes, including full backward compatibility
notes for code written against v1.0.7.